In [ ]:
# -*- coding: utf-8 -*-
"""
COMPARAÇÃO COMPLETA ENTRE 4 MÉTODOS DE COMPENSAÇÃO TÉRMICA:

1) RF Direto ponto a ponto
2) RF + Features
3) Linear + Features
4) Park (1999)

Métricas únicas:
- RMSD vs referência
- CCDM vs referência

Autor: Luiz Eduardo Abdala José
Versão organizada e unificada
"""

import re
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore", category=UserWarning)


# ========================================================================
# =========================== PARÂMETROS GERAIS ==========================
# ========================================================================

ARQ_BASE = "base-completo--.pkl"
REF_TEMP = 30

FREQ_MIN_KHZ = 40
FREQ_MAX_KHZ = 50
SMOOTH_WIN = 5    # média móvel (ímpar)

RF_COMP_POINT_PARAMS = dict(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=2,
    min_samples_split=4,
    max_features="sqrt",
    n_jobs=-1,
    random_state=0,
)

RF_COMP_FEAT_PARAMS = dict(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=3,
    min_samples_split=4,
    n_jobs=-1,
    random_state=42,
)

CAPS = dict(gain_frac=0.60, offset_frac=0.60, tilt_frac=0.40)

TAU_MAX_FRAC = 0.025
ANCHOR_TO_REF_ENDS = True

PARK_MAX_SHIFT_FRAC = 0.25
PARK_OVERLAP_MIN = 0.60
PARK_SMOOTH_WIN = 5
PARK_NSTEPS = 201


# ========================================================================
# =========================== FUNÇÕES GERAIS =============================
# ========================================================================

def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None and fmin_khz <= f/1e3 <= fmax_khz:
            cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs)[order]


def moving_average(arr, win):
    if win <= 1 or win % 2 == 0:
        return arr.copy()
    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode="edge")
    kernel = np.ones(win) / win
    smooth = np.convolve(arr_pad, kernel, mode="valid")
    if len(smooth) > len(arr):
        smooth = smooth[:len(arr)]
    elif len(smooth) < len(arr):
        smooth = np.pad(smooth, (0, len(arr) - len(smooth)), mode="edge")
    return smooth


def slope_over_band(f, x):
    return float((x[-1] - x[0]) / (f[-1] - f[0] + 1e-12))


def energy_weighted_centroid(f, x):
    w = x * x
    den = float(np.trapezoid(w, f))
    if den <= 1e-18:
        return float(np.mean(f))
    num = float(np.trapezoid(f * w, f))
    return num / den


def shift_interp(x, f, tau):
    f_shift = f + tau
    return np.interp(f, f_shift, x, left=x[0], right=x[-1])


def add_extra_features_matrix(X):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    amp = (X.max(axis=1) - X.min(axis=1)).reshape(-1, 1)
    return np.hstack([X, mu, sd, amp])


def compute_simple_features(X, f):
    N = X.shape[0]
    feats = np.zeros((N, 5))
    for i in range(N):
        x = X[i]
        feats[i] = [
            np.mean(x),
            np.std(x),
            x.max() - x.min(),
            slope_over_band(f, x),
            energy_weighted_centroid(f, x)
        ]
    return feats, ["mean", "std", "amp", "slope", "centroid"]


def rmsd(y, ref):
    return float(np.sqrt(np.mean((y - ref)**2)))


def ccdm(y, ref):
    y0 = y - np.mean(y)
    r0 = ref - np.mean(ref)
    num = float(np.sum(y0 * r0))
    den = float(np.sqrt(np.sum(y0**2) * np.sum(r0**2))) + 1e-18
    corr = num/den
    return float(1 - corr)


def escolher_temperaturas_dano(df, dano, n=6):
    temps = df.loc[df["falha"] == dano, "temperatura_c"].unique()
    temps = np.sort(temps)
    if len(temps) == 0:
        return np.array([])
    if len(temps) <= n:
        return temps
    return np.random.choice(temps, size=n, replace=False)


# ========================================================================
# ====================== MÉTODOS I: RF DIRETO ============================
# ========================================================================

def compensar_rf_direto(df, fcols, fHz):
    df_sem = df[df["falha"] == 0]
    X_sem = df_sem[fcols].to_numpy(float)
    T_sem = df_sem["temperatura_c"].to_numpy(float)

    ref_pool = df_sem.loc[np.isclose(df_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)
    if len(ref_pool) > 0:
        y_ref = np.median(ref_pool, axis=0)
    else:
        y_ref = np.median(X_sem, axis=0)

    Y_target = (y_ref[None, :] - X_sem)

    X_aug = add_extra_features_matrix(X_sem)
    X_in = np.hstack([X_aug, T_sem.reshape(-1,1)])

    rf = RandomForestRegressor(**RF_COMP_POINT_PARAMS)
    rf.fit(X_in, Y_target)

    X_all = df[fcols].to_numpy(float)
    T_all = df["temperatura_c"].to_numpy(float)
    X_aug_all = add_extra_features_matrix(X_all)
    X_in_all = np.hstack([X_aug_all, T_all.reshape(-1,1)])

    Y_comp = X_all + rf.predict(X_in_all)

    for i in range(len(Y_comp)):
        Y_comp[i] = moving_average(Y_comp[i], SMOOTH_WIN)

    df2 = df.copy()
    df2[fcols] = Y_comp
    return df2, y_ref


# ========================================================================
# ====================== MÉTODOS II: RF FEATURES =========================
# ========================================================================

def fit_feature_models_rf(F, T):
    models = {}
    T = T.reshape(-1, 1)
    for j, name in enumerate(["mean", "std", "amp", "slope", "centroid"]):
        rf = RandomForestRegressor(**RF_COMP_FEAT_PARAMS)
        rf.fit(T, F[:, j])
        models[name] = rf
    return models


def targets_at_ref(models):
    Tref = np.array([[REF_TEMP]])
    return {name: float(m.predict(Tref)[0]) for name, m in models.items()}


def apply_physical_comp(x, f, targets, caps, ref):
    x = x.copy()
    mean_t = targets["mean"]
    amp_t = targets["amp"]
    slope_t = targets["slope"]
    cent_t = targets["centroid"]

    mean_x = np.mean(x)
    amp_x = x.max() - x.min()
    slope_x = slope_over_band(f, x)

    off = mean_t - mean_x
    off = np.clip(off, -caps["offset_frac"]*amp_x, caps["offset_frac"]*amp_x)
    x = x + off

    gain = amp_t / max(amp_x, 1e-9)
    gain = np.clip(gain, 1 - caps["gain_frac"], 1 + caps["gain_frac"])
    x = mean_t + gain*(x - mean_t)

    delta_s = slope_t - slope_x
    dfb = (f[-1] - f[0])
    tilt = (delta_s * dfb) * np.linspace(-0.5, 0.5, len(x))
    tilt = np.clip(tilt, -caps["tilt_frac"]*amp_x, caps["tilt_frac"]*amp_x)
    x = x + tilt

    cent_x = energy_weighted_centroid(f, x)
    delta_c = cent_t - cent_x
    tau = np.clip(delta_c, -TAU_MAX_FRAC*dfb, TAU_MAX_FRAC*dfb)
    x = shift_interp(x, f, tau)

    if ANCHOR_TO_REF_ENDS:
        e0 = x[0] - ref[0]
        e1 = x[-1] - ref[-1]
        corr = np.linspace(e0, e1, len(x))
        x = x - corr

    return x


def compensar_rf_features(df, fcols, fHz):
    df_sem = df[df["falha"] == 0]
    X_sem = df_sem[fcols].to_numpy(float)
    T_sem = df_sem["temperatura_c"].to_numpy(float)

    F_sem, names = compute_simple_features(X_sem, fHz)
    models = fit_feature_models_rf(F_sem, T_sem)
    targets = targets_at_ref(models)

    pool = df_sem.loc[np.isclose(df_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)
    y_ref = np.median(pool, axis=0) if len(pool)>0 else np.median(X_sem, axis=0)

    X_all = df[fcols].to_numpy(float)
    Y = np.zeros_like(X_all)
    for i in range(len(X_all)):
        y = apply_physical_comp(X_all[i], fHz, targets, CAPS, y_ref)
        Y[i] = moving_average(y, SMOOTH_WIN)

    df2 = df.copy()
    df2[fcols] = Y
    return df2, y_ref


# ========================================================================
# ====================== MÉTODOS III: LINEAR FEATURES ====================
# ========================================================================

def fit_feature_models_linear(F, T):
    models = {}
    T = T.reshape(-1, 1)
    for j, name in enumerate(["mean", "std", "amp", "slope", "centroid"]):
        lr = LinearRegression()
        lr.fit(T, F[:, j])
        models[name] = lr
    return models


def compensar_linear_features(df, fcols, fHz):
    df_sem = df[df["falha"] == 0]
    X_sem = df_sem[fcols].to_numpy(float)
    T_sem = df_sem["temperatura_c"].to_numpy(float)

    F_sem, names = compute_simple_features(X_sem, fHz)
    models = fit_feature_models_linear(F_sem, T_sem)
    targets = targets_at_ref(models)

    pool = df_sem.loc[np.isclose(df_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)
    y_ref = np.median(pool, axis=0) if len(pool)>0 else np.median(X_sem, axis=0)

    X_all = df[fcols].to_numpy(float)
    Y = np.zeros_like(X_all)
    for i in range(len(X_all)):
        y = apply_physical_comp(X_all[i], fHz, targets, CAPS, y_ref)
        Y[i] = moving_average(y, SMOOTH_WIN)

    df2 = df.copy()
    df2[fcols] = Y
    return df2, y_ref


# ========================================================================
# ====================== MÉTODOS IV: PARK ===============================
# ========================================================================

def park_single(x, ref, fHz):
    n = len(x)
    df_band = fHz[-1] - fHz[0]
    tau_max = PARK_MAX_SHIFT_FRAC * df_band

    best_err = 1e99
    best_tau, best_dS = 0, 0

    for tau in np.linspace(-tau_max, tau_max, PARK_NSTEPS):
        x_shift = shift_interp(x, fHz, tau)
        dS = np.mean(ref - x_shift)
        err = np.sum((ref - (x_shift + dS))**2)
        if err < best_err:
            best_err = err
            best_tau, best_dS = tau, dS

    y = shift_interp(x, fHz, best_tau) + best_dS
    y = moving_average(y, PARK_SMOOTH_WIN)
    return y


def compensar_park(df, fcols, fHz):
    df_sem = df[df["falha"] == 0]
    X_sem = df_sem[fcols].to_numpy(float)

    pool = df_sem.loc[np.isclose(df_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)
    y_ref = np.median(pool, axis=0) if len(pool)>0 else np.median(X_sem, axis=0)

    X_all = df[fcols].to_numpy(float)
    Y = np.zeros_like(X_all)
    for i in range(len(X_all)):
        Y[i] = park_single(X_all[i], y_ref, fHz)

    df2 = df.copy()
    df2[fcols] = Y
    return df2, y_ref


# ========================================================================
# ====================== MÉTRICAS & PLOTS ===============================
# ========================================================================

def calcular_metricas(df, fcols, y_ref):
    X = df[fcols].to_numpy(float)
    df2 = df.copy()
    df2["RMSD"] = [rmsd(x, y_ref) for x in X]
    df2["CCDM"] = [ccdm(x, y_ref) for x in X]
    return df2


# ========================================================================
# ============================== EXECUÇÃO DAS IA =========================
# ========================================================================

def executar_compensacoes():
    print("🔹 Carregando base...")
    df = pd.read_pickle(ARQ_BASE)

    fcols, fhz = get_freq_columns(df, FREQ_MIN_KHZ, FREQ_MAX_KHZ)

    # 4 métodos:
    df_rf, y1 = compensar_rf_direto(df, fcols, fhz)
    df_feat, y2 = compensar_rf_features(df, fcols, fhz)
    df_lin, y3 = compensar_linear_features(df, fcols, fhz)
    df_park, y4 = compensar_park(df, fcols, fhz)

    # métricas
    df_rf = calcular_metricas(df_rf, fcols, y1)
    df_feat = calcular_metricas(df_feat, fcols, y2)
    df_lin = calcular_metricas(df_lin, fcols, y3)
    df_park = calcular_metricas(df_park, fcols, y4)

    print("✅ Compensações e métricas concluídas.")

    return df_rf, df_feat, df_lin, df_park


# Rode APENAS quando quiser recalcular tudo:
df_rf, df_feat, df_lin, df_park = executar_compensacoes()


In [ ]:
# ==============================================
# =========== BIBLIOTECAS ======================
# ==============================================
import numpy as np
import matplotlib.pyplot as plt
import os

# ============================================================
# =========== CONFIGURAÇÃO DO SALVAMENTO AUTOMÁTICO ==========
# ============================================================

# Usa sua pasta já existente
PASTA_PRINCIPAL = "resultados_comp"

if not os.path.isdir(PASTA_PRINCIPAL):
    raise RuntimeError("⚠️ A pasta 'resultados_comp' não existe! Crie antes de rodar o código.")

# Criar subpasta baseada em REF_TEMP e faixa de frequência
PASTA_SAIDA = os.path.join(
    PASTA_PRINCIPAL,
    f"REF{REF_TEMP}C_{int(FREQ_MIN_KHZ)}-{int(FREQ_MAX_KHZ)}kHz"
)
os.makedirs(PASTA_SAIDA, exist_ok=True)

# Guardar função original
_original_show = plt.show

# Contador global
_global_fig_counter = 1

# ============================================================
# =========== FUNÇÕES DE MÉTRICAS (RMSD e CCDM) ==============
# ============================================================

def calc_rmsd(ref, cur):
    """
    RMSD entre dois sinais (ref e cur).
    """
    ref = np.array(ref)
    cur = np.array(cur)
    return np.sqrt(np.mean((cur - ref)**2))


def calc_ccdm(ref, cur):
    """
    CCDM usando correlação cruzada normalizada.
    """
    ref = (ref - np.mean(ref)) / np.std(ref)
    cur = (cur - np.mean(cur)) / np.std(cur)
    corr = np.correlate(ref, cur, mode='full')
    max_corr = np.max(np.abs(corr))
    return 1 - max_corr/len(ref)


# =====================================================================
# === FUNÇÃO PRINCIPAL: PLOTAR RMSD E CCDM PARA UMA TEMPERATURA ========
# =====================================================================

def plot_metrics_for_temperature(
    data_dict,
    temperature,
    methods_to_plot=None,
    ref_method="RF_PONTO_A_PONTO"
):
    """
    Plota RMSD e CCDM dos métodos desejados para uma temperatura específica.
    """

    # Verifica se temperatura existe no dicionário
    if temperature not in data_dict:
        raise ValueError(f"Temperatura {temperature}°C não encontrada no dataset.")

    # Espectro de referência
    ref = data_dict[temperature][ref_method]

    # Quais métodos o usuário quer
    available_methods = list(data_dict[temperature].keys())

    if methods_to_plot is None:
        methods_to_plot = available_methods
    else:
        methods_to_plot = [m for m in methods_to_plot if m in available_methods]

    rmsd_vals = []
    ccdm_vals = []

    for method in methods_to_plot:
        cur = data_dict[temperature][method]

        rmsd_vals.append(calc_rmsd(ref, cur))
        ccdm_vals.append(calc_ccdm(ref, cur))

    # ===============================
    # ======= GRÁFICO RMSD ==========
    # ===============================
    plt.figure(figsize=(10,5))
    plt.bar(methods_to_plot, rmsd_vals, color='royalblue')
    plt.title(f"RMSD por Método – Temperatura {temperature}°C")
    plt.ylabel("RMSD")
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0)
    plt.tight_layout()
    plt.show()

    # ===============================
    # ======= GRÁFICO CCDM ==========
    # ===============================
    plt.figure(figsize=(10,5))
    plt.bar(methods_to_plot, ccdm_vals, color='darkorange')
    plt.title(f"CCDM por Método – Temperatura {temperature}°C")
    plt.ylabel("CCDM")
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0)
    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def histograma_rf_park_temperaturas_validas_2painel(df_rf, df_park, n_temps=6, seed=42):
    # =========================
    # 1) Achar temperaturas válidas
    # =========================
    temps_rf_d0 = set(df_rf.loc[df_rf["falha"] == 0, "temperatura_c"].unique())
    temps_rf_d1 = set(df_rf.loc[df_rf["falha"] == 1, "temperatura_c"].unique())
    temps_rf_d2 = set(df_rf.loc[df_rf["falha"] == 2, "temperatura_c"].unique())
    temps_rf_validas = temps_rf_d0 & temps_rf_d1 & temps_rf_d2

    temps_pk_d0 = set(df_park.loc[df_park["falha"] == 0, "temperatura_c"].unique())
    temps_pk_d1 = set(df_park.loc[df_park["falha"] == 1, "temperatura_c"].unique())
    temps_pk_d2 = set(df_park.loc[df_park["falha"] == 2, "temperatura_c"].unique())
    temps_pk_validas = temps_pk_d0 & temps_pk_d1 & temps_pk_d2

    temps_validas = sorted(list(temps_rf_validas & temps_pk_validas))

    if len(temps_validas) == 0:
        print("❌ Nenhuma temperatura possui os 3 danos em ambos os métodos!")
        return

    # =========================
    # 2) Reduzir quantidade de temperaturas
    # =========================
    if len(temps_validas) > n_temps:
        rng = np.random.default_rng(seed)
        temps_validas = sorted(rng.choice(temps_validas, n_temps, replace=False))

    danos = [0, 1, 2]
    cores = ["tab:blue", "tab:orange", "tab:red"]

    # =========================
    # 3) Layout das barras
    # =========================
    n_t = len(temps_validas)
    x = np.arange(n_t)

    bar_w = 0.10
    gap = 0.08
    rf_offsets = np.array([0, 1, 2]) * bar_w
    pk_offsets = (3 * bar_w + gap) + np.array([0, 1, 2]) * bar_w
    x_center = x + ((rf_offsets.mean() + pk_offsets.mean()) / 2.0)

    # =========================
    # 4) Coletar médias
    # =========================
    def coletar_medias(df, metric_name):
        out = {d: [] for d in danos}
        for d in danos:
            for T in temps_validas:
                mask = (df["falha"] == d) & np.isclose(df["temperatura_c"], T)
                out[d].append(df.loc[mask, metric_name].mean() if np.any(mask) else np.nan)
        return out

    rmsd_rf = coletar_medias(df_rf, "RMSD")
    rmsd_pk = coletar_medias(df_park, "RMSD")
    ccdm_rf = coletar_medias(df_rf, "CCDM")
    ccdm_pk = coletar_medias(df_park, "CCDM")

    # =========================
    # 5) Configuração visual
    # =========================
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 13,
        "axes.titlesize": 14,
        "axes.labelsize": 13,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12
    })

    fig, axes = plt.subplots(
        nrows=1, ncols=2,
        figsize=(16, 5.8),
        dpi=300
    )

    fig.patch.set_facecolor("white")

    for ax in axes:
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(0.8)
        ax.spines["bottom"].set_linewidth(0.8)
        ax.tick_params(axis='both', which='major', length=4, width=0.8)

    # =========================
    # 6) Painel (a): RMSD
    # =========================
    ax = axes[0]
    for i, d in enumerate(danos):
        ax.bar(
            x + rf_offsets[i], rmsd_rf[d],
            width=bar_w,
            color=cores[i],
            alpha=0.55,
            edgecolor="black",
            linewidth=0.5,
            label=f"RF Direto — Dano {d}"
        )
        ax.bar(
            x + pk_offsets[i], rmsd_pk[d],
            width=bar_w,
            color=cores[i],
            alpha=1.0,
            edgecolor="black",
            linewidth=0.5,
            label=f"Park — Dano {d}"
        )

    ax.set_title("(a) RMSD", pad=10)
    ax.set_xlabel("Temperatura (°C)")
    ax.set_ylabel("RMSD")
    ax.set_xticks(x_center)
    ax.set_xticklabels([f"{int(t)}" if float(t).is_integer() else f"{t:.1f}" for t in temps_validas])

    # =========================
    # 7) Painel (b): CCDM
    # =========================
    ax = axes[1]
    for i, d in enumerate(danos):
        ax.bar(
            x + rf_offsets[i], ccdm_rf[d],
            width=bar_w,
            color=cores[i],
            alpha=0.55,
            edgecolor="black",
            linewidth=0.5
        )
        ax.bar(
            x + pk_offsets[i], ccdm_pk[d],
            width=bar_w,
            color=cores[i],
            alpha=1.0,
            edgecolor="black",
            linewidth=0.5
        )

    ax.set_title("(b) CCDM", pad=10)
    ax.set_xlabel("Temperatura (°C)")
    ax.set_ylabel("CCDM")
    ax.set_xticks(x_center)
    ax.set_xticklabels([f"{int(t)}" if float(t).is_integer() else f"{t:.1f}" for t in temps_validas])

    # =========================
    # 8) Legenda global limpa
    # =========================
    handles, labels = axes[0].get_legend_handles_labels()

    seen = set()
    uniq = []
    for h, lab in zip(handles, labels):
        if lab not in seen:
            uniq.append((h, lab))
            seen.add(lab)

    handles_u, labels_u = zip(*uniq)

    fig.legend(
        handles_u, labels_u,
        loc="upper center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 1.06),
        fontsize=13,
        handlelength=1.8,
        columnspacing=1.5
    )

    plt.tight_layout(rect=[0, 0, 1, 0.90])
    plt.show()


histograma_rf_park_temperaturas_validas_2painel(df_rf, df_park, n_temps=8)

In [ ]:
# =====================================================================
# ============ HISTOGRAMAS ARTIGO COM TEMPERATURAS FIXAS ==============
# =====================================================================

import matplotlib.pyplot as plt
import numpy as np

def gerar_histogramas_artigo_fixo(df_rf, df_feat, df_lin, df_park):

    # -----------------------------------------------------------------
    # CONFIGURAÇÃO GLOBAL (FONTE MAIOR)
    # -----------------------------------------------------------------
    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": 14,
        "axes.titlesize": 16,
        "axes.labelsize": 15,
        "legend.fontsize": 14,
        "xtick.labelsize": 13,
        "ytick.labelsize": 13
    })

    df_dict = {
        "RF Ponto-a-ponto": df_rf,
        "RF Features": df_feat,
        "Linear Features": df_lin,
        "Park": df_park,
    }

    metodos_labels = ["RF Ponto-a-ponto", "RF Features", "Linear Features", "Park"]

    metodos_cores = {
        "RF Ponto-a-ponto": "#1f77b4",
        "RF Features": "#ff7f0e",
        "Linear Features": "#2ca02c",
        "Park": "#d62728"
    }

    temperaturas_fixas = np.array([-10, 10, 20, 30, 40, 50, 60, 70])

    dano = 0
    metricas = ["RMSD", "CCDM"]

    fig, axes = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(16, 6),
        frameon=True,
        dpi=300
    )

    fig.patch.set_facecolor("white")
    fig.patch.set_edgecolor("none")

    bar_width = 0.18
    n_t = len(temperaturas_fixas)
    n_m = len(metodos_labels)
    x_base = np.arange(n_t)

    for j, metric_name in enumerate(metricas):

        ax = axes[j]
        vals = np.zeros((n_m, n_t))

        for m_idx, metodo in enumerate(metodos_labels):

            dfm = df_dict[metodo]

            for t_idx, T in enumerate(temperaturas_fixas):

                mask = (
                    (dfm["falha"] == dano) &
                    (np.isclose(dfm["temperatura_c"], T))
                )

                if np.any(mask):
                    vals[m_idx, t_idx] = dfm.loc[mask, metric_name].mean()
                else:
                    vals[m_idx, t_idx] = np.nan

        # Plot barras
        for m_idx, metodo in enumerate(metodos_labels):
            ax.bar(
                x_base + m_idx * bar_width,
                vals[m_idx],
                width=bar_width,
                color=metodos_cores[metodo],
                edgecolor="black",
                linewidth=0.6,
                label=metodo
            )

        ax.set_xticks(x_base + (n_m - 1) * bar_width / 2)
        ax.set_xticklabels(temperaturas_fixas)

        ax.set_xlabel("Temperature (°C)")
        ax.set_ylabel(metric_name)

        if metric_name == "RMSD":
            ax.set_title("(a) RMSD — Damage 0", pad=12)
        else:
            ax.set_title("(b) CCDM — Damage 0", pad=12)

        ax.grid(False)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(1)
        ax.spines["bottom"].set_linewidth(1)

        ax.tick_params(axis="both", which="major", length=5, width=1)

    # Legenda global
    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=4,
        frameon=False,
        bbox_to_anchor=(0.5, 1.05)
    )

    plt.tight_layout(rect=[0, 0, 1, 0.92])

    plt.savefig(
        "Histogramas_SHM_TemperaturasFixas_Dano0.tiff",
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()

    print("📊 Figura com 2 gráficos (dano 0) gerada com sucesso!")

# executar
gerar_histogramas_artigo_fixo(df_rf, df_feat, df_lin, df_park)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def histograma_rf_park_temperaturas_validas_2painel(df_rf, df_park, n_temps=6, seed=42):
    # =========================
    # 1) Achar temperaturas válidas
    # =========================
    temps_rf_d0 = set(df_rf.loc[df_rf["falha"] == 0, "temperatura_c"].unique())
    temps_rf_d1 = set(df_rf.loc[df_rf["falha"] == 1, "temperatura_c"].unique())
    temps_rf_d2 = set(df_rf.loc[df_rf["falha"] == 2, "temperatura_c"].unique())
    temps_rf_validas = temps_rf_d0 & temps_rf_d1 & temps_rf_d2

    temps_pk_d0 = set(df_park.loc[df_park["falha"] == 0, "temperatura_c"].unique())
    temps_pk_d1 = set(df_park.loc[df_park["falha"] == 1, "temperatura_c"].unique())
    temps_pk_d2 = set(df_park.loc[df_park["falha"] == 2, "temperatura_c"].unique())
    temps_pk_validas = temps_pk_d0 & temps_pk_d1 & temps_pk_d2

    temps_validas = sorted(list(temps_rf_validas & temps_pk_validas))

    if len(temps_validas) == 0:
        print("❌ Nenhuma temperatura possui os 3 danos em ambos os métodos!")
        return

    # =========================
    # 2) Reduzir quantidade de temperaturas
    # =========================
    if len(temps_validas) > n_temps:
        rng = np.random.default_rng(seed)
        temps_validas = sorted(rng.choice(temps_validas, n_temps, replace=False))

    danos = [0, 1, 2]
    cores = ["tab:blue", "tab:orange", "tab:red"]

    # =========================
    # 3) Layout das barras
    # =========================
    n_t = len(temps_validas)
    x = np.arange(n_t)

    bar_w = 0.10
    gap = 0.08
    rf_offsets = np.array([0, 1, 2]) * bar_w
    pk_offsets = (3 * bar_w + gap) + np.array([0, 1, 2]) * bar_w
    x_center = x + ((rf_offsets.mean() + pk_offsets.mean()) / 2.0)

    # =========================
    # 4) Coletar médias
    # =========================
    def coletar_medias(df, metric_name):
        out = {d: [] for d in danos}
        for d in danos:
            for T in temps_validas:
                mask = (df["falha"] == d) & np.isclose(df["temperatura_c"], T)
                out[d].append(df.loc[mask, metric_name].mean() if np.any(mask) else np.nan)
        return out

    rmsd_rf = coletar_medias(df_rf, "RMSD")
    rmsd_pk = coletar_medias(df_park, "RMSD")
    ccdm_rf = coletar_medias(df_rf, "CCDM")
    ccdm_pk = coletar_medias(df_park, "CCDM")

    # =========================
    # 5) Configuração visual
    # =========================
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 16,
        "axes.titlesize": 17,
        "axes.labelsize": 16,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "legend.fontsize": 14
    })

    fig, axes = plt.subplots(
        nrows=1, ncols=2,
        figsize=(17, 6.2),
        dpi=300
    )

    fig.patch.set_facecolor("white")

    for ax in axes:
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(0.9)
        ax.spines["bottom"].set_linewidth(0.9)
        ax.tick_params(axis='both', which='major', length=4.5, width=0.9)

    # =========================
    # 6) Painel (a): RMSD
    # =========================
    ax = axes[0]
    for i, d in enumerate(danos):
        ax.bar(
            x + rf_offsets[i], rmsd_rf[d],
            width=bar_w,
            color=cores[i],
            alpha=0.55,
            edgecolor="black",
            linewidth=0.6,
            label=f"RF Direto — Dano {d}"
        )
        ax.bar(
            x + pk_offsets[i], rmsd_pk[d],
            width=bar_w,
            color=cores[i],
            alpha=1.0,
            edgecolor="black",
            linewidth=0.6,
            label=f"Park — Dano {d}"
        )

    ax.set_title("(a) RMSD", pad=10)
    ax.set_xlabel("Temperatura (°C)")
    ax.set_ylabel("RMSD")
    ax.set_xticks(x_center)
    ax.set_xticklabels([f"{int(t)}" if float(t).is_integer() else f"{t:.1f}" for t in temps_validas])

    # =========================
    # 7) Painel (b): CCDM
    # =========================
    ax = axes[1]
    for i, d in enumerate(danos):
        ax.bar(
            x + rf_offsets[i], ccdm_rf[d],
            width=bar_w,
            color=cores[i],
            alpha=0.55,
            edgecolor="black",
            linewidth=0.6
        )
        ax.bar(
            x + pk_offsets[i], ccdm_pk[d],
            width=bar_w,
            color=cores[i],
            alpha=1.0,
            edgecolor="black",
            linewidth=0.6
        )

    ax.set_title("(b) CCDM", pad=10)
    ax.set_xlabel("Temperatura (°C)")
    ax.set_ylabel("CCDM")
    ax.set_xticks(x_center)
    ax.set_xticklabels([f"{int(t)}" if float(t).is_integer() else f"{t:.1f}" for t in temps_validas])

    # =========================
    # 8) Legenda global limpa
    # =========================
    handles, labels = axes[0].get_legend_handles_labels()

    seen = set()
    uniq = []
    for h, lab in zip(handles, labels):
        if lab not in seen:
            uniq.append((h, lab))
            seen.add(lab)

    handles_u, labels_u = zip(*uniq)

    fig.legend(
        handles_u, labels_u,
        loc="upper center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 1.06),
        fontsize=14,
        handlelength=1.8,
        columnspacing=1.5
    )

    plt.tight_layout(rect=[0, 0, 1, 0.90])
    plt.show()


def gerar_histogramas_artigo_fixo(df_rf, df_feat, df_lin, df_park):
    # -----------------------------------------------------------------
    # CONFIGURAÇÃO GLOBAL
    # -----------------------------------------------------------------
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 16,
        "axes.titlesize": 17,
        "axes.labelsize": 16,
        "legend.fontsize": 14,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14
    })

    df_dict = {
        "RF Ponto-a-ponto": df_rf,
        "RF Features": df_feat,
        "Linear Features": df_lin,
        "Park": df_park,
    }

    metodos_labels = ["RF Ponto-a-ponto", "RF Features", "Linear Features", "Park"]

    metodos_cores = {
        "RF Ponto-a-ponto": "#1f77b4",
        "RF Features": "#ff7f0e",
        "Linear Features": "#2ca02c",
        "Park": "#d62728"
    }

    temperaturas_fixas = np.array([-10, 10, 20, 30, 40, 50, 60, 70])

    dano = 0
    metricas = ["RMSD", "CCDM"]

    fig, axes = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(17, 6.2),
        dpi=300
    )

    fig.patch.set_facecolor("white")

    bar_width = 0.18
    n_t = len(temperaturas_fixas)
    n_m = len(metodos_labels)
    x_base = np.arange(n_t)

    for j, metric_name in enumerate(metricas):
        ax = axes[j]
        vals = np.zeros((n_m, n_t))

        for m_idx, metodo in enumerate(metodos_labels):
            dfm = df_dict[metodo]

            for t_idx, T in enumerate(temperaturas_fixas):
                mask = (
                    (dfm["falha"] == dano) &
                    (np.isclose(dfm["temperatura_c"], T))
                )

                if np.any(mask):
                    vals[m_idx, t_idx] = dfm.loc[mask, metric_name].mean()
                else:
                    vals[m_idx, t_idx] = np.nan

        for m_idx, metodo in enumerate(metodos_labels):
            ax.bar(
                x_base + m_idx * bar_width,
                vals[m_idx],
                width=bar_width,
                color=metodos_cores[metodo],
                edgecolor="black",
                linewidth=0.7,
                label=metodo
            )

        ax.set_xticks(x_base + (n_m - 1) * bar_width / 2)
        ax.set_xticklabels(temperaturas_fixas)

        ax.set_xlabel("Temperatura (°C)")
        ax.set_ylabel(metric_name)

        if metric_name == "RMSD":
            ax.set_title("(a) RMSD — Dano 0", pad=12)
        else:
            ax.set_title("(b) CCDM — Dano 0", pad=12)

        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(1.0)
        ax.spines["bottom"].set_linewidth(1.0)
        ax.tick_params(axis="both", which="major", length=5, width=1.0)

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=4,
        frameon=False,
        bbox_to_anchor=(0.5, 1.05),
        fontsize=14
    )

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()


# executar
histograma_rf_park_temperaturas_validas_2painel(df_rf, df_park, n_temps=8)
gerar_histogramas_artigo_fixo(df_rf, df_feat, df_lin, df_park)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# CONFIGURAÇÃO GLOBAL DE FONTE
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 22,
    "axes.labelsize": 22,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20
})


# ============================================================
# GRÁFICO 1 — RF DIRETO vs PARK
# ============================================================

def histograma_rf_park_temperaturas_validas_2painel(df_rf, df_park, n_temps=6, seed=42):

    temps_rf_d0 = set(df_rf.loc[df_rf["falha"] == 0, "temperatura_c"].unique())
    temps_rf_d1 = set(df_rf.loc[df_rf["falha"] == 1, "temperatura_c"].unique())
    temps_rf_d2 = set(df_rf.loc[df_rf["falha"] == 2, "temperatura_c"].unique())

    temps_pk_d0 = set(df_park.loc[df_park["falha"] == 0, "temperatura_c"].unique())
    temps_pk_d1 = set(df_park.loc[df_park["falha"] == 1, "temperatura_c"].unique())
    temps_pk_d2 = set(df_park.loc[df_park["falha"] == 2, "temperatura_c"].unique())

    temps_validas = sorted(list(
        (temps_rf_d0 & temps_rf_d1 & temps_rf_d2) &
        (temps_pk_d0 & temps_pk_d1 & temps_pk_d2)
    ))

    if len(temps_validas) == 0:
        print("Nenhuma temperatura possui os três danos em ambos os métodos.")
        return

    if len(temps_validas) > n_temps:
        rng = np.random.default_rng(seed)
        temps_validas = sorted(rng.choice(temps_validas, n_temps, replace=False))

    danos = [0, 1, 2]
    cores = ["tab:blue", "tab:orange", "tab:red"]

    x = np.arange(len(temps_validas))

    bar_w = 0.12
    gap = 0.10

    rf_offsets = np.array([0, 1, 2]) * bar_w
    pk_offsets = (3 * bar_w + gap) + np.array([0, 1, 2]) * bar_w

    x_center = x + ((rf_offsets.mean() + pk_offsets.mean()) / 2)

    def medias(df, metrica):
        out = {d: [] for d in danos}

        for d in danos:
            for T in temps_validas:
                mask = (df["falha"] == d) & np.isclose(df["temperatura_c"], T)

                if np.any(mask):
                    out[d].append(df.loc[mask, metrica].mean())
                else:
                    out[d].append(np.nan)

        return out

    rmsd_rf = medias(df_rf, "RMSD")
    rmsd_pk = medias(df_park, "RMSD")

    ccdm_rf = medias(df_rf, "CCDM")
    ccdm_pk = medias(df_park, "CCDM")

    fig, axes = plt.subplots(1, 2, figsize=(20, 7.2), dpi=300)

    for ax in axes:
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # ======================
    # RMSD
    # ======================
    ax = axes[0]

    for i, d in enumerate(danos):
        ax.bar(
            x + rf_offsets[i], rmsd_rf[d],
            width=bar_w,
            color=cores[i],
            alpha=0.55,
            edgecolor="black",
            linewidth=0.8,
            label=f"RF direto — dano {d}"
        )

        ax.bar(
            x + pk_offsets[i], rmsd_pk[d],
            width=bar_w,
            color=cores[i],
            alpha=1,
            edgecolor="black",
            linewidth=0.8,
            label=f"Park — dano {d}"
        )

    ax.set_ylabel("RMSD")
    ax.set_xlabel("Temperatura (°C)", labelpad=10)

    ax.set_xticks(x_center)
    ax.set_xticklabels([
        f"{int(t)}" if float(t).is_integer() else f"{t:.1f}"
        for t in temps_validas
    ])

    ax.text(
        0.5, -0.30,
        "(a) RMSD",
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=22
    )

    # ======================
    # CCDM
    # ======================
    ax = axes[1]

    for i, d in enumerate(danos):
        ax.bar(
            x + rf_offsets[i], ccdm_rf[d],
            width=bar_w,
            color=cores[i],
            alpha=0.55,
            edgecolor="black",
            linewidth=0.8
        )

        ax.bar(
            x + pk_offsets[i], ccdm_pk[d],
            width=bar_w,
            color=cores[i],
            alpha=1,
            edgecolor="black",
            linewidth=0.8
        )

    ax.set_ylabel("CCDM")
    ax.set_xlabel("Temperatura (°C)", labelpad=10)

    ax.set_xticks(x_center)
    ax.set_xticklabels([
        f"{int(t)}" if float(t).is_integer() else f"{t:.1f}"
        for t in temps_validas
    ])

    ax.text(
        0.5, -0.30,
        "(b) CCDM",
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=22
    )

    handles, labels = axes[0].get_legend_handles_labels()

    # remover duplicatas mantendo ordem
    vistos = set()
    handles_unicos = []
    labels_unicos = []
    for h, lab in zip(handles, labels):
        if lab not in vistos:
            handles_unicos.append(h)
            labels_unicos.append(lab)
            vistos.add(lab)

    fig.legend(
        handles_unicos,
        labels_unicos,
        loc="upper center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 1.06)
    )

    plt.tight_layout(rect=[0, 0.10, 1, 0.92])
    plt.show()


# ============================================================
# GRÁFICO 2 — COMPARAÇÃO GLOBAL
# ============================================================

def gerar_histogramas_artigo_fixo(df_rf, df_feat, df_lin, df_park):

    df_dict = {
        "RF ponto-a-ponto": df_rf,
        "RF por features": df_feat,
        "Regressão linear": df_lin,
        "Park": df_park
    }

    metodos = list(df_dict.keys())

    cores = {
        "RF ponto-a-ponto": "#1f77b4",
        "RF por features": "#ff7f0e",
        "Regressão linear": "#2ca02c",
        "Park": "#d62728"
    }

    temperaturas = np.array([-10, 10, 20, 30, 40, 50, 60, 70])
    metricas = ["RMSD", "CCDM"]

    fig, axes = plt.subplots(1, 2, figsize=(20, 7.2), dpi=300)

    largura = 0.18
    x = np.arange(len(temperaturas))

    for j, metrica in enumerate(metricas):
        ax = axes[j]

        valores = np.zeros((len(metodos), len(temperaturas)))

        for m, metodo in enumerate(metodos):
            dfm = df_dict[metodo]

            for i, T in enumerate(temperaturas):
                mask = (dfm["falha"] == 0) & np.isclose(dfm["temperatura_c"], T)

                if np.any(mask):
                    valores[m, i] = dfm.loc[mask, metrica].mean()
                else:
                    valores[m, i] = np.nan

        for m, metodo in enumerate(metodos):
            ax.bar(
                x + m * largura,
                valores[m],
                width=largura,
                color=cores[metodo],
                edgecolor="black",
                linewidth=0.8,
                label=metodo
            )

        ax.set_xticks(x + (len(metodos) - 1) * largura / 2)
        ax.set_xticklabels(temperaturas)

        ax.set_xlabel("Temperatura (°C)", labelpad=10)
        ax.set_ylabel(metrica)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(False)

        if metrica == "RMSD":
            ax.text(
                0.5, -0.30,
                "(a) RMSD — dano 0",
                transform=ax.transAxes,
                ha="center",
                va="top",
                fontsize=22
            )
        else:
            ax.text(
                0.5, -0.30,
                "(b) CCDM — dano 0",
                transform=ax.transAxes,
                ha="center",
                va="top",
                fontsize=22
            )

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=4,
        frameon=False,
        bbox_to_anchor=(0.5, 1.06)
    )

    plt.tight_layout(rect=[0, 0.10, 1, 0.92])
    plt.show()


# ============================================================
# EXECUTAR
# ============================================================

histograma_rf_park_temperaturas_validas_2painel(df_rf, df_park, n_temps=8)

gerar_histogramas_artigo_fixo(df_rf, df_feat, df_lin, df_park)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# GLOBAL FONT SETTINGS
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 22,
    "axes.labelsize": 22,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20
})


# ============================================================
# FIGURE 1 — DIRECT RF vs PARK
# ============================================================

def histogram_rf_park_valid_temperatures_2panel(df_rf, df_park, n_temps=6, seed=42):

    temps_rf_d0 = set(df_rf.loc[df_rf["falha"] == 0, "temperatura_c"].unique())
    temps_rf_d1 = set(df_rf.loc[df_rf["falha"] == 1, "temperatura_c"].unique())
    temps_rf_d2 = set(df_rf.loc[df_rf["falha"] == 2, "temperatura_c"].unique())

    temps_pk_d0 = set(df_park.loc[df_park["falha"] == 0, "temperatura_c"].unique())
    temps_pk_d1 = set(df_park.loc[df_park["falha"] == 1, "temperatura_c"].unique())
    temps_pk_d2 = set(df_park.loc[df_park["falha"] == 2, "temperatura_c"].unique())

    valid_temps = sorted(list(
        (temps_rf_d0 & temps_rf_d1 & temps_rf_d2) &
        (temps_pk_d0 & temps_pk_d1 & temps_pk_d2)
    ))

    if len(valid_temps) == 0:
        print("No temperature contains all three damage states in both methods.")
        return

    if len(valid_temps) > n_temps:
        rng = np.random.default_rng(seed)
        valid_temps = sorted(rng.choice(valid_temps, n_temps, replace=False))

    damages = [0, 1, 2]
    colors = ["tab:blue", "tab:orange", "tab:red"]

    x = np.arange(len(valid_temps))

    bar_w = 0.12
    gap = 0.10

    rf_offsets = np.array([0, 1, 2]) * bar_w
    pk_offsets = (3 * bar_w + gap) + np.array([0, 1, 2]) * bar_w

    x_center = x + ((rf_offsets.mean() + pk_offsets.mean()) / 2)

    def averages(df, metric):
        out = {d: [] for d in damages}

        for d in damages:
            for T in valid_temps:
                mask = (df["falha"] == d) & np.isclose(df["temperatura_c"], T)

                if np.any(mask):
                    out[d].append(df.loc[mask, metric].mean())
                else:
                    out[d].append(np.nan)

        return out

    rmsd_rf = averages(df_rf, "RMSD")
    rmsd_pk = averages(df_park, "RMSD")

    ccdm_rf = averages(df_rf, "CCDM")
    ccdm_pk = averages(df_park, "CCDM")

    fig, axes = plt.subplots(1, 2, figsize=(20, 7.2), dpi=300)

    for ax in axes:
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # ======================
    # RMSD
    # ======================
    ax = axes[0]

    for i, d in enumerate(damages):
        ax.bar(
            x + rf_offsets[i], rmsd_rf[d],
            width=bar_w,
            color=colors[i],
            alpha=0.55,
            edgecolor="black",
            linewidth=0.8,
            label=f"Direct RF — damage {d}"
        )

        ax.bar(
            x + pk_offsets[i], rmsd_pk[d],
            width=bar_w,
            color=colors[i],
            alpha=1,
            edgecolor="black",
            linewidth=0.8,
            label=f"Park — damage {d}"
        )

    ax.set_ylabel("RMSD")
    ax.set_xlabel("Temperature (°C)", labelpad=10)

    ax.set_xticks(x_center)
    ax.set_xticklabels([
        f"{int(t)}" if float(t).is_integer() else f"{t:.1f}"
        for t in valid_temps
    ])

    ax.text(
        0.5, -0.30,
        "(a) RMSD",
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=22
    )

    # ======================
    # CCDM
    # ======================
    ax = axes[1]

    for i, d in enumerate(damages):
        ax.bar(
            x + rf_offsets[i], ccdm_rf[d],
            width=bar_w,
            color=colors[i],
            alpha=0.55,
            edgecolor="black",
            linewidth=0.8
        )

        ax.bar(
            x + pk_offsets[i], ccdm_pk[d],
            width=bar_w,
            color=colors[i],
            alpha=1,
            edgecolor="black",
            linewidth=0.8
        )

    ax.set_ylabel("CCDM")
    ax.set_xlabel("Temperature (°C)", labelpad=10)

    ax.set_xticks(x_center)
    ax.set_xticklabels([
        f"{int(t)}" if float(t).is_integer() else f"{t:.1f}"
        for t in valid_temps
    ])

    ax.text(
        0.5, -0.30,
        "(b) CCDM",
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=22
    )

    handles, labels = axes[0].get_legend_handles_labels()

    seen = set()
    unique_handles = []
    unique_labels = []
    for h, lab in zip(handles, labels):
        if lab not in seen:
            unique_handles.append(h)
            unique_labels.append(lab)
            seen.add(lab)

    fig.legend(
        unique_handles,
        unique_labels,
        loc="upper center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 1.06)
    )

    plt.tight_layout(rect=[0, 0.10, 1, 0.92])
    plt.show()


# ============================================================
# FIGURE 2 — GLOBAL COMPARISON
# ============================================================

def generate_article_histograms_fixed(df_rf, df_feat, df_lin, df_park):

    df_dict = {
        "Point-wise RF": df_rf,
        "Feature-based RF": df_feat,
        "Linear Regression": df_lin,
        "Park": df_park
    }

    methods = list(df_dict.keys())

    colors = {
        "Point-wise RF": "#1f77b4",
        "Feature-based RF": "#ff7f0e",
        "Linear Regression": "#2ca02c",
        "Park": "#d62728"
    }

    temperatures = np.array([-10, 10, 20, 30, 40, 50, 60, 70])
    metrics = ["RMSD", "CCDM"]

    fig, axes = plt.subplots(1, 2, figsize=(20, 7.2), dpi=300)

    width = 0.18
    x = np.arange(len(temperatures))

    for j, metric in enumerate(metrics):
        ax = axes[j]

        values = np.zeros((len(methods), len(temperatures)))

        for m, method in enumerate(methods):
            dfm = df_dict[method]

            for i, T in enumerate(temperatures):
                mask = (dfm["falha"] == 0) & np.isclose(dfm["temperatura_c"], T)

                if np.any(mask):
                    values[m, i] = dfm.loc[mask, metric].mean()
                else:
                    values[m, i] = np.nan

        for m, method in enumerate(methods):
            ax.bar(
                x + m * width,
                values[m],
                width=width,
                color=colors[method],
                edgecolor="black",
                linewidth=0.8,
                label=method
            )

        ax.set_xticks(x + (len(methods) - 1) * width / 2)
        ax.set_xticklabels(temperatures)

        ax.set_xlabel("Temperature (°C)", labelpad=10)
        ax.set_ylabel(metric)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(False)

        if metric == "RMSD":
            ax.text(
                0.5, -0.30,
                "(a) RMSD — damage 0",
                transform=ax.transAxes,
                ha="center",
                va="top",
                fontsize=22
            )
        else:
            ax.text(
                0.5, -0.30,
                "(b) CCDM — damage 0",
                transform=ax.transAxes,
                ha="center",
                va="top",
                fontsize=22
            )

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=4,
        frameon=False,
        bbox_to_anchor=(0.5, 1.06)
    )

    plt.tight_layout(rect=[0, 0.10, 1, 0.92])
    plt.show()


# ============================================================
# RUN
# ============================================================

histogram_rf_park_valid_temperatures_2panel(df_rf, df_park, n_temps=8)

generate_article_histograms_fixed(df_rf, df_feat, df_lin, df_park)